# Trustworthy Diabetes Risk-Screening Decision System
## CDC BRFSS 2015 Machine Learning Pipeline & Trustworthy AI Audit

### Project Overview
This notebook implements an academic research decision system for diabetes risk screening using the **CDC Behavioral Risk Factor Surveillance System (BRFSS) 2015** balanced dataset (70,692 original records, 21 survey features, binary target).

The project develops and rigorously compares **two predictive models**:
1. **Model 1: Baseline Logistic Regression** — A standard machine learning pipeline with feature scaling, thresholding at 0.5, and standard performance metrics.
2. **Model 2: Trust-Aware Calibrated Logistic Regression** — Logistic Regression augmented with probability calibration (Platt scaling / sigmoid), SHAP explainability, Fairlearn demographic fairness audits, calibration reliability analysis, controlled input perturbation robustness testing, and post-processing fairness mitigation.

### Critical Methodological Standards
- **Two-Model Relationship**: Model 1 is the uncalibrated Logistic Regression baseline. Model 2 uses the **exact same underlying Logistic Regression** with sigmoid probability calibration.
- **Evaluation vs Modification**: SHAP, fairness auditing, and robustness testing are **evaluation components**, not automatic modifications to the model.
- **No Inherent Superiority Claim**: We do not claim or assume that Model 2 is automatically more accurate, fair, or robust. The core objective is to investigate whether trustworthy AI evaluation provides actionable insights into model reliability, demographic disparities, and operational limitations.
- **Strict Leakage Prevention**: Stratified 70% Train, 15% Validation, and 15% Test split.
- **Validation-Only Calibration**: `CalibratedClassifierCV` is fitted strictly on the 15% Validation split using `FrozenEstimator` (scikit-learn >= 1.6). The 15% Test split is held out purely for final independent evaluation.
- **No Training on All Records**: Evaluated models are trained strictly on the 70% training split (48,339 records).
- **Academic Disclaimer**: Predictions represent survey-based positive class probabilities and do NOT constitute clinical diagnosis or real-world population prevalence.


## 1. Environment Setup & Dependency Installation

This cell configures the runtime environment for Google Colab or local execution.
Required packages: `scikit-learn>=1.6.0` (required for `FrozenEstimator`), `shap>=0.44.0`, `fairlearn>=0.9.0`, `plotly>=5.18.0`, `pandas>=2.0.0`, `numpy>=1.24.0`, `matplotlib>=3.7.0`, `seaborn>=0.12.0`, `joblib>=1.3.0`.


In [ ]:
# Check environment and install dependencies if running in Google Colab
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab runtime.")
    get_ipython().system("pip install -q scikit-learn>=1.6.0 shap>=0.44.0 fairlearn>=0.9.0 plotly>=5.18.0 joblib>=1.3.0")
except (ImportError, NameError):
    IN_COLAB = False
    print("Running on local Python environment.")

# Add project root directory to sys.path to import src modules
current_dir = os.getcwd()
if os.path.basename(current_dir) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, ".."))
else:
    PROJECT_ROOT = os.path.abspath(current_dir)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root set to: {PROJECT_ROOT}")


### 1.1 Package Imports and Seed Configuration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings("ignore")

import sklearn
print(f"Scikit-learn version in runtime: {sklearn.__version__}")

# Set plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# Import existing modular src package
from src import (
    data_loader,
    preprocessing,
    model_baseline,
    model_trustworthy,
    explainability,
    fairness,
    calibration,
    robustness,
    mitigation,
    artifact_io
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Successfully imported all src modules with random seed = 42.")


## 2. Dataset Loading, Codebook Verification & Leakage Checks

The dataset is derived from the **CDC BRFSS 2015** survey with an initial 50/50 balance between respondents with no diabetes (0) and those with prediabetes or diabetes (1).

We inspect:
- Dataset dimensions, data types, and missing value counts (confirmed 0).
- Feature value ranges against official CDC BRFSS 2015 codebook definitions.
- Target correlation check to ensure no target leakage (correlation < 0.95).


In [ ]:
csv_path = os.path.join(PROJECT_ROOT, "data", "diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
raw_df = data_loader.load_and_validate_dataset(csv_path)

print(f"Raw dataset shape: {raw_df.shape}")
print(f"Target distribution:
{raw_df['Diabetes_binary'].value_counts()}")


### 2.1 Feature Definitions and Label Mappings (Data Dictionary)


In [ ]:
data_dict = data_loader.get_data_dictionary()
data_dict_table = pd.DataFrame([
    {
        "Feature": feat,
        "Type": info["type"],
        "Valid Range": info["valid_range"],
        "Description": info["description"]
    }
    for feat, info in data_dict.items()
])
print(f"Total features documented: {len(data_dict_table)}")
data_dict_table.head(10)


## 3. Data Cleaning: Duplicate Records Investigation & Methodology

The raw dataset contains 70,692 rows, of which **1,635 rows** are exact duplicates across all 22 columns (involving 2,803 total rows).

### Critical Research Question: Accidental Copies or Valid Survey Responses?
In large public health surveys such as BRFSS:
1. **Discrete Survey Space**: The questionnaire records responses across 21 discretized variables (14 binary flags, discrete integer BMI, 5 General Health categories, 13 Age brackets, 8 Income brackets). Across 70,000+ respondents, identical discrete profiles (e.g., young, normal-weight, non-smoking, high-income individuals in excellent health) naturally arise among independent survey respondents.
2. **Label Conflict**: Examining the identical 21-feature profiles reveals **408 distinct feature vectors (involving 1,001 rows) where respondents have conflicting targets** (one respondent reported Diabetes=0, while another reported Diabetes=1). This directly proves that identical feature vectors represent **distinct survey respondents** with different clinical outcomes, rather than accidental database duplication.
3. **Cross-Split Contamination**: However, if identical rows are retained during random train/test splitting, identical profiles will populate both the training and test sets. The model's evaluation would then partly reflect memorization of identical records rather than true out-of-sample generalization.

### Empirical Performance Comparison (With vs Without Deduplication)
We compare baseline and calibrated models trained with and without deduplication:


In [ ]:
## 3. Methodological Investigation: Duplicate Records & Dataset Design

The raw dataset contains 70,692 rows, of which **1,635 rows** are exact duplicates across all 22 columns (involving 2,803 total rows).

### In-Depth Investigation Findings:
1. **Nature of Duplicates**: The BRFSS questionnaire records self-reported discrete values across 21 variables. Across 70,000+ respondents, common demographic/lifestyle profiles naturally produce identical discrete response vectors.
2. **Conflicting Target Labels**: We identified **408 distinct feature vectors (involving 1,001 rows) where respondents have conflicting targets** (one respondent reported Diabetes=0, while another identical profile reported Diabetes=1). This directly proves that identical feature rows represent **distinct survey respondents with different underlying clinical states**, rather than accidental database duplication.
3. **Primary vs Secondary Dataset Strategy**:
   - **Primary Dataset (70,692 records)**: Preserves all surveyed respondents and the balanced 50/50 design.
   - **Secondary Sensitivity Dataset (69,057 records)**: Removes the 1,635 exact identical rows to evaluate whether model performance is sensitive to identical response vectors.
   - **Methodological Limitation**: Deduplication removes identical rows across all 22 columns, but does **not** guarantee feature-level isolation between training and testing (due to the 408 conflicting profiles). Neither approach guarantees complete respondent-level isolation; both are rigorously documented.


## 4. Stratified Data Splitting (70% Train / 15% Validation / 15% Test)

To ensure strict leak-free evaluation:
- **Training Set (70%, 48,339 rows)**: Used exclusively to fit the preprocessing transformers and train the Logistic Regression classifiers.
- **Validation Set (15%, 10,359 rows)**: Used exclusively to fit the sigmoid calibration function (`CalibratedClassifierCV`) and test threshold tuning.
- **Test Set (15%, 10,359 rows)**: **Held out strictly.** Never touched during training, scaling, feature selection, or calibration fitting. All final metrics, fairness audits, calibration curves, and sample cases are drawn from this set.


In [ ]:
# PRIMARY DATASET SPLIT (70,692 records - exact 50/50 balance)
X_train, X_val, X_test, y_train, y_val, y_test, feature_names = data_loader.split_dataset(
    raw_df, random_state=RANDOM_SEED
)

# Convert to DataFrame with original feature names for tabular explainability and fairness
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_val_df = pd.DataFrame(X_val, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

print('Primary dataset stratified splitting completed successfully.')


## 5. Preprocessing Pipeline Architecture

We use a Scikit-Learn `ColumnTransformer`:
- **Continuous & Ordinal Features (7)**: `BMI`, `MentHlth`, `PhysHlth`, `GenHlth`, `Age`, `Education`, `Income` are normalized using `StandardScaler`.
- **Binary Features (14)**: `HighBP`, `HighChol`, `CholCheck`, `Smoker`, `Stroke`, `HeartDiseaseorAttack`, `PhysActivity`, `Fruits`, `Veggies`, `HvyAlcoholConsump`, `AnyHealthcare`, `NoDocbcCost`, `DiffWalk`, `Sex` are passed through without modification.


In [ ]:
preprocessor = preprocessing.build_preprocessor()
feature_order = preprocessing.get_feature_order_after_transform()
print(f"Feature transformation order ({len(feature_order)} columns):
{feature_order}")


## 6. Model 1: Baseline Logistic Regression

Model 1 is the uncalibrated Scikit-Learn Pipeline combining the `ColumnTransformer` with `LogisticRegression(max_iter=1000, solver="lbfgs", random_state=42)`.
Standard metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC, Brier Score) are computed on the held-out test set at the default threshold (0.5).


In [ ]:
print("--- Training Model 1: Baseline Logistic Regression ---")
baseline_model = model_baseline.train_baseline_model(X_train, y_train, random_state=RANDOM_SEED)

print("--- Evaluating Model 1 on Held-out Test Set ---")
baseline_metrics = model_baseline.evaluate_model(
    baseline_model, X_test, y_test, threshold=0.5, model_name="Model 1 (Baseline)"
)

print(f"Accuracy:  {baseline_metrics['accuracy']:.4f}")
print(f"Precision: {baseline_metrics['precision']:.4f}")
print(f"Recall:    {baseline_metrics['recall']:.4f}")
print(f"F1-Score:  {baseline_metrics['f1']:.4f}")
print(f"ROC-AUC:   {baseline_metrics['roc_auc']:.4f}")
print(f"Brier:     {baseline_metrics['brier_score']:.4f}")


## 7. Model 2: Trust-Aware Calibrated Logistic Regression

### Calibration Methodology (scikit-learn >= 1.6 FrozenEstimator)
In modern Scikit-Learn (>= 1.6), `cv='prefit'` is deprecated/removed in favor of `sklearn.frozen.FrozenEstimator`.
The probability calibration methodology follows 4 strict steps:
1. **Fit Inner Pipeline**: Preprocessing + Logistic Regression is trained on `X_train, y_train` (70%).
2. **Freeze Weights**: Wrap the fitted pipeline in `FrozenEstimator(inner_pipeline)`.
3. **Fit Calibration**: `CalibratedClassifierCV(estimator=frozen_pipeline, method='sigmoid')` is fitted strictly on `X_val, y_val` (15%).
4. **Independent Evaluation**: Evaluated only on the untouched test set (`X_test, y_test`, 15%).

Platt scaling learns parameters $A$ and $B$ via maximum likelihood on validation data:
$$P(Y=1 \mid f(x)) = \frac{1}{1 + \exp(A f(x) + B)}$$


In [ ]:
print("--- Training Model 2: Trust-Aware Calibrated Logistic Regression ---")
trustworthy_model = model_trustworthy.train_trustworthy_model(
    X_train, y_train, X_val, y_val, random_state=RANDOM_SEED
)

print("--- Evaluating Model 2 on Held-out Test Set ---")
trustworthy_metrics = model_trustworthy.evaluate_trustworthy_model(
    trustworthy_model, X_test, y_test, threshold=0.5
)

print(f"Accuracy:  {trustworthy_metrics['accuracy']:.4f}")
print(f"Precision: {trustworthy_metrics['precision']:.4f}")
print(f"Recall:    {trustworthy_metrics['recall']:.4f}")
print(f"F1-Score:  {trustworthy_metrics['f1']:.4f}")
print(f"ROC-AUC:   {trustworthy_metrics['roc_auc']:.4f}")
print(f"Brier:     {trustworthy_metrics['brier_score']:.4f}")


### 7.1 Model Comparison & Trustworthy AI Principles

> **Methodological Principle**: We do not claim that Model 2 is automatically more accurate, fair, or robust than Model 1.
> Both models share the exact same underlying Logistic Regression coefficients. Model 2 merely applies sigmoid probability calibration.
> Trustworthy AI components (SHAP, Fairlearn, Reliability diagrams, Perturbation tests) are **diagnostic evaluations** designed to inspect model behaviour, group disparities, and operational bounds.


In [ ]:
# Primary Model Comparison
comparison_df = pd.DataFrame([
    {'Metric': 'Accuracy', 'Model 1 (Baseline)': baseline_metrics['accuracy'], 'Model 2 (Trust-Aware)': trustworthy_metrics['accuracy'], 'Difference': trustworthy_metrics['accuracy'] - baseline_metrics['accuracy']},
    {'Metric': 'Precision', 'Model 1 (Baseline)': baseline_metrics['precision'], 'Model 2 (Trust-Aware)': trustworthy_metrics['precision'], 'Difference': trustworthy_metrics['precision'] - baseline_metrics['precision']},
    {'Metric': 'Recall', 'Model 1 (Baseline)': baseline_metrics['recall'], 'Model 2 (Trust-Aware)': trustworthy_metrics['recall'], 'Difference': trustworthy_metrics['recall'] - baseline_metrics['recall']},
    {'Metric': 'F1-Score', 'Model 1 (Baseline)': baseline_metrics['f1'], 'Model 2 (Trust-Aware)': trustworthy_metrics['f1'], 'Difference': trustworthy_metrics['f1'] - baseline_metrics['f1']},
    {'Metric': 'ROC-AUC', 'Model 1 (Baseline)': baseline_metrics['roc_auc'], 'Model 2 (Trust-Aware)': trustworthy_metrics['roc_auc'], 'Difference': trustworthy_metrics['roc_auc'] - baseline_metrics['roc_auc']},
    {'Metric': 'Brier Score', 'Model 1 (Baseline)': baseline_metrics['brier_score'], 'Model 2 (Trust-Aware)': trustworthy_metrics['brier_score'], 'Difference': trustworthy_metrics['brier_score'] - baseline_metrics['brier_score']}
])
print('=== PRIMARY EXPERIMENTAL RESULTS (70,692 records) ===')
print(comparison_df.round(4).to_string(index=False))

# Secondary Sensitivity Experiment on Deduplicated Data (69,057 records)
clean_df = data_loader.remove_duplicates(raw_df)
X_tr_cln, X_v_cln, X_te_cln, y_tr_cln, y_v_cln, y_te_cln, _ = data_loader.split_dataset(clean_df, random_state=RANDOM_SEED)
m1_cln = model_baseline.train_baseline_model(X_tr_cln, y_tr_cln, random_state=RANDOM_SEED)
e1_cln = model_baseline.evaluate_model(m1_cln, X_te_cln, y_te_cln, threshold=0.5)

sec_comp_df = pd.DataFrame([
    {'Metric': 'Accuracy', 'Primary (Raw 70k)': baseline_metrics['accuracy'], 'Secondary (Dedup 69k)': e1_cln['accuracy'], 'Delta': e1_cln['accuracy'] - baseline_metrics['accuracy']},
    {'Metric': 'F1-Score', 'Primary (Raw 70k)': baseline_metrics['f1'], 'Secondary (Dedup 69k)': e1_cln['f1'], 'Delta': e1_cln['f1'] - baseline_metrics['f1']},
    {'Metric': 'ROC-AUC', 'Primary (Raw 70k)': baseline_metrics['roc_auc'], 'Secondary (Dedup 69k)': e1_cln['roc_auc'], 'Delta': e1_cln['roc_auc'] - baseline_metrics['roc_auc']},
    {'Metric': 'Brier Score', 'Primary (Raw 70k)': baseline_metrics['brier_score'], 'Secondary (Dedup 69k)': e1_cln['brier_score'], 'Delta': e1_cln['brier_score'] - baseline_metrics['brier_score']}
])
print('\n=== SECONDARY SENSITIVITY EXPERIMENT COMPARISON ===')
print(sec_comp_df.round(4).to_string(index=False))


## 8. Trustworthy AI Pillar 1: SHAP Explainability

### Technical Disclosure: Log-Odds vs Calibrated Probabilities
`CalibratedClassifierCV` wraps the inner Scikit-Learn `Pipeline`. SHAP explanation is computed using `shap.LinearExplainer` applied to the **inner Logistic Regression classifier** in the standardized feature space:
- SHAP values explain the **uncalibrated log-odds score** $f(x) = w^T x + b$.
- Positive SHAP values indicate features pushing toward diabetes risk, while negative values push toward low risk.
- They do **not** directly represent additive contributions to the final sigmoid-calibrated probability.
- Background dataset: 200 training samples. Explanation set: 500 test samples.


In [ ]:
print("--- Computing Global SHAP Feature Importance ---")
shap_values, explained_features = explainability.compute_global_shap(
    trustworthy_model,
    X_train_df,
    feature_order,
    n_background=200,
    n_explain=500,
    random_state=RANDOM_SEED
)

global_importance_df = explainability.compute_global_importance(shap_values, explained_features)
print("Top 10 Features by Global SHAP Importance:")
print(global_importance_df.head(10).to_string(index=False))

# Plot top features
plt.figure(figsize=(10, 6))
sns.barplot(data=global_importance_df.head(10), x="mean_abs_shap", y="feature", palette="Blues_r")
plt.title("Top 10 Feature Contributions (Mean |SHAP Value| on Log-Odds)")
plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 9. Trustworthy AI Pillar 2: Fairness Audit (Fairlearn)

Fairness is audited across three protected demographic attributes available in the BRFSS survey:
1. **Sex**: Female (0) vs Male (1)
2. **Age**: 13 age bands (1: 18-24 to 13: 80+)
3. **Income**: 8 income brackets (1: <$10k to 8: $75k+)

*Note: No race/ethnicity variable is present in this BRFSS-derived dataset.*
Metrics evaluated per subgroup:
- **TPR (True Positive Rate)**: $\frac{TP}{TP + FN}$
- **FPR (False Positive Rate)**: $\frac{FP}{FP + TN}$
- **FNR (False Negative Rate)**: $1 - TPR$
- **Selection Rate**: $\frac{TP + FP}{N}$
- **Group Sample Size ($N$)**
- **Demographic Parity Difference**: $\max(SR) - \min(SR)$
- **Equalized Odds Difference**: $\max(|TPR_{diff}|, |FPR_{diff}|)$


In [ ]:
print("--- Computing Fairness Metrics Across Sex, Age, and Income ---")
fairness_dict = fairness.compute_fairness_both_models(
    baseline_preds=baseline_metrics["y_pred"],
    trustworthy_preds=trustworthy_metrics["y_pred"],
    y_true=y_test,
    y_prob_base=baseline_metrics["y_prob"],
    y_prob_trust=trustworthy_metrics["y_prob"],
    X_test_df=X_test_df
)

print("Fairness disparity summary:")
for attr, comp in fairness_dict["comparison_summary"].items():
    print(f"Attribute: {attr.upper()}")
    print(f"  Demographic Parity Diff (Baseline):    {comp['demographic_parity_diff_baseline']:.4f}")
    print(f"  Demographic Parity Diff (Trustworthy): {comp['demographic_parity_diff_trustworthy']:.4f}")
    print(f"  Equalized Odds Diff (Baseline):        {comp['equalized_odds_diff_baseline']:.4f}")
    print(f"  Equalized Odds Diff (Trustworthy):     {comp['equalized_odds_diff_trustworthy']:.4f}")


## 10. Trustworthy AI Pillar 3: Calibration & Reliability

Calibration evaluates whether a predicted probability of $p$ corresponds to an observed event frequency of $p$.
- **Reliability Diagram**: Compares mean predicted value against observed fraction of positives across 10 uniform probability bins.
- **Brier Score**: Mean squared error between predicted probability and binary outcome.
- **Expected Calibration Error (ECE)**: Bin-weighted average absolute calibration error:
  $$\text{ECE} = \sum_{b=1}^B \frac{|B_b|}{N} |\text{acc}(B_b) - \text{conf}(B_b)|$$

*Prevalence Disclaimer: The dataset was constructed with an artificial 50/50 balance. Predicted probabilities reflect this balanced research cohort and must NOT be interpreted as real-world population disease prevalence.*


In [ ]:
print("--- Computing Calibration Curves and ECE ---")
calibration_dict = calibration.compute_calibration_both_models(
    y_test=y_test,
    y_prob_baseline=baseline_metrics["y_prob"],
    y_prob_trustworthy=trustworthy_metrics["y_prob"],
    n_bins=10
)

b_ece = calibration_dict["baseline_calibration"]["ece"]
t_ece = calibration_dict["trustworthy_calibration"]["ece"]
print(f"Baseline Model ECE:    {b_ece:.4f} (Brier: {baseline_metrics['brier_score']:.4f})")
print(f"Trustworthy Model ECE: {t_ece:.4f} (Brier: {trustworthy_metrics['brier_score']:.4f})")
print(f"ECE Relative Change:   {(t_ece - b_ece)/b_ece * 100:+.2f}%")


## 11. Trustworthy AI Pillar 4: Controlled Robustness & Sensitivity Testing

We conduct controlled sensitivity experiments on 500 test samples by perturbing single features within valid BRFSS ranges:
1. `bmi_plus1`: BMI increased by 1 (clipped max 98).
2. `bmi_minus1`: BMI decreased by 1 (clipped min 12).
3. `genhlth_worse`: GenHlth increased by 1 level (worse, clipped max 5).
4. `genhlth_better`: GenHlth decreased by 1 level (better, clipped min 1).
5. `physhlth_plus1`: PhysHlth increased by 1 day (clipped max 30).
6. `menthlth_plus1`: MentHlth increased by 1 day (clipped max 30).

We measure:
- Mean absolute probability change $|\Delta p|$.
- Prediction flip rate at threshold 0.5.


In [ ]:
print("--- Running Controlled Perturbation Robustness Tests ---")
robustness_results_df = robustness.run_all_perturbations(
    baseline_model=baseline_model,
    trustworthy_model=trustworthy_model,
    X_test_df=X_test_df,
    y_test=y_test,
    n_samples=500,
    random_state=RANDOM_SEED
)

robustness_summary_df = robustness.compute_robustness_summary(robustness_results_df)
print("Robustness Sensitivity Summary:")
print(robustness_summary_df.to_string(index=False))


## 12. Sample Case Selection (CASE_01 to CASE_15)

We select exactly 15 representative patient cases from the held-out test dataset:
- 8 negative cases (`Diabetes_binary` = 0)
- 7 positive cases (`Diabetes_binary` = 1)
For each case, we record feature values, actual status, baseline prediction/probability, trustworthy prediction/probability, and model correctness.
Local SHAP explanation values are computed for these cases.


In [ ]:
print("--- Extracting 15 Test Sample Cases ---")
# Stratified selection from held-out test set
test_indices_0 = np.where(y_test == 0)[0]
test_indices_1 = np.where(y_test == 1)[0]

selected_idx_0 = test_indices_0[:8]
selected_idx_1 = test_indices_1[:7]
selected_indices = np.concatenate([selected_idx_0, selected_idx_1])

sample_cases_list = []
case_ids = [f"CASE_{i+1:02d}" for i in range(15)]

for i, idx in enumerate(selected_indices):
    case_id = case_ids[i]
    row_features = X_test_df.iloc[idx].to_dict()
    actual_label = int(y_test[idx])
    
    m1_prob = float(baseline_metrics["y_prob"][idx])
    m1_pred = int(baseline_metrics["y_pred"][idx])
    m2_prob = float(trustworthy_metrics["y_prob"][idx])
    m2_pred = int(trustworthy_metrics["y_pred"][idx])
    
    case_data = {
        "case_id": case_id,
        "actual_label": actual_label,
        "model1_pred": m1_pred,
        "model1_prob": round(m1_prob, 4),
        "model1_correct": (m1_pred == actual_label),
        "model2_pred": m2_pred,
        "model2_prob": round(m2_prob, 4),
        "model2_correct": (m2_pred == actual_label),
    }
    case_data.update(row_features)
    sample_cases_list.append(case_data)

sample_cases_df = pd.DataFrame(sample_cases_list)
print(f"Extracted {len(sample_cases_df)} sample cases.")

# Compute local SHAP values for these 15 cases
sample_records_df = X_test_df.iloc[selected_indices]
local_shap_values, base_value = explainability.compute_local_shap(
    trustworthy_model,
    sample_records_df,
    feature_order,
    X_train_df,
    n_background=200
)
print(f"Computed local SHAP values for {local_shap_values.shape[0]} cases.")


## 13. Experimental Fairness Mitigation (Fairlearn ThresholdOptimizer)

When demographic audits identify group disparities (e.g., across Income brackets), post-processing mitigations can optimize group-specific decision thresholds to satisfy equalized odds constraints.

### Methodological Protocol
- **Validation Fitting Only**: `ThresholdOptimizer` is fitted strictly on validation data (`X_val, y_val, X_val_df['Income']`).
- **Untouched Test Evaluation**: Performance and disparity impacts are measured on untouched test data (`X_test, y_test, X_test_df['Income']`).
- **Original Models Untouched**: The primary Baseline and Calibrated models remain unchanged.
- **Reporting**: Full before-and-after fairness metrics, subgroup TPR/FPR, and utility trade-offs.


In [ ]:
print('--- Running Post-Processing Fairness Mitigation Experiment on Income ---')
# Fitted strictly on validation data, evaluated on untouched test data
mitigation_results = mitigation.run_mitigation_experiment(
    baseline_model=baseline_model,
    X_val=X_val,
    y_val=y_val,
    X_val_df=X_val_df,
    X_test=X_test,
    y_test=y_test,
    X_test_df=X_test_df,
    sensitive_col='Income',
    random_state=RANDOM_SEED
)

mitigation_report = mitigation.format_mitigation_report(mitigation_results)
print(mitigation_report)


## 14. Artifact Serialization & Complete Verification

All 11 required project artifacts are saved to `artifacts/`:
1. `baseline_model.joblib`: Fitted Baseline Pipeline
2. `trustworthy_model.joblib`: Fitted Calibrated Classifier
3. `model_metadata.json`: Architecture, split specs, and feature schema
4. `evaluation_metrics.json`: Standard classification metrics for both models
5. `fairness_metrics.json`: Group-level metrics across Sex, Age, and Income
6. `robustness_results.csv`: Granular record-level perturbation logs
7. `robustness_summary.csv`: Aggregated sensitivity summary
8. `sample_cases.csv`: 15 test records with dual-model predictions
9. `shap_global_importance.csv`: Global SHAP feature ranking
10. `shap_individual_values.csv`: Local SHAP values for 15 test cases
11. `calibration_data.csv`: Calibration curve coordinates and class-conditional distributions


In [ ]:
artifacts_dir = os.path.join(PROJECT_ROOT, "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)

# 1. Models
artifact_io.save_model(baseline_model, "baseline_model.joblib", artifacts_dir=artifacts_dir)
artifact_io.save_model(trustworthy_model, "trustworthy_model.joblib", artifacts_dir=artifacts_dir)

# 2. Metadata & Metrics
artifact_io.save_model_metadata(baseline_metrics, trustworthy_metrics, feature_names, artifacts_dir=artifacts_dir)
artifact_io.save_evaluation_metrics(baseline_metrics, trustworthy_metrics, artifacts_dir=artifacts_dir)

# 3. Fairness
fairness.save_fairness_artifacts(fairness_dict, artifacts_dir=artifacts_dir)

# 4. Calibration
calibration.save_calibration_artifacts(calibration_dict, artifacts_dir=artifacts_dir)

# 5. Robustness
robustness.save_robustness_artifacts(robustness_results_df, robustness_summary_df, artifacts_dir=artifacts_dir)

# 6. Sample Cases
artifact_io.save_sample_cases(sample_cases_df, artifacts_dir=artifacts_dir)

# 7. SHAP
explainability.save_shap_artifacts(global_importance_df, local_shap_values, feature_order, case_ids, artifacts_dir=artifacts_dir)

print("--- Verifying Artifact Reloading ---")
loaded_artifacts = artifact_io.load_all_artifacts(artifacts_dir=artifacts_dir)
for k, v in loaded_artifacts.items():
    status = "OK" if v is not None else "MISSING"
    print(f"  Artifact [{k}]: {status}")

all_loaded = all(v is not None for v in loaded_artifacts.values())
if all_loaded:
    print("
>>> ALL 11 ARTIFACTS SUCCESSFULLY GENERATED AND VERIFIED! <<<")
else:
    print("
>>> WARNING: SOME ARTIFACTS WERE NOT LOADED! <<<")


## 15. Medical and Research Limitations

1. **Non-Clinical Screening Demonstration**: This system is developed for academic exploration of trustworthy machine learning principles. It is not clinically validated and must not be used for diagnostic or treatment decisions.
2. **BRFSS Survey Nature**: Features are derived from telephone survey self-reports, subject to recall bias and reporting variance.
3. **Prevalence Distortion**: The 50/50 balance in the dataset does not match population diabetes prevalence (~10-14% in US adults). Predicted probabilities must be interpreted relative to this balanced cohort.
4. **Explainability Bounds**: SHAP identifies statistical associations within the trained model, not biological etiology or clinical causality.
5. **Mitigation Limitations**: Post-processing threshold optimization reduces statistical disparity on the training/validation distribution but does not address systemic health inequities present in observational data.
